## TUGAS MANDIRI (2320506038-Debi Aprilia)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 12 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7, 9, dan 10) ingin memiliki alat bantu **prediksi omzet harian** tiap cabang, agar dapat merencanakan stok bahan baku dengan lebih baik. Anda ditugaskan membangun model regresi untuk memprediksi `total_penjualan` sebuah pesanan berdasarkan atribut-atributnya.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat dataset.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


### Instruksi Pengerjaan

Buat notebook baru **`Tugas11_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, baca `tugas11_penjualan.csv`, lalu kerjakan bagian **A sampai E** berikut:


In [2]:
# Mengimpor library yang dibutuhkan
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Membuat SparkSession
spark = SparkSession.builder \
    .appName("Tugas11-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

# Mengatur tampilan log
spark.sparkContext.setLogLevel("ERROR")

# Mengatur jumlah partisi
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

26/11/18 21:12:08 WARN Utils: Your hostname, deb-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/11/18 21:12:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/18 21:12:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession berhasil dibuat!
Versi Spark: 3.5.9


In [3]:
# Membaca dataset CSV
df = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)

# Menampilkan lima data pertama
df.show(5)

# Menampilkan struktur kolom dan tipe data
df.printSchema()

# Menghitung jumlah seluruh data
print("Total data:", df.count())

+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows

root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- total_penjualan: double (nullable = true)

Total data: 2500


**A. Feature Engineering** *(bobot 20%)*

Gunakan `StringIndexer` untuk mengubah kolom `nama_menu`, `cabang`, dan `waktu_pesan` menjadi indeks numerik. Gabungkan seluruh fitur (`qty`, `harga_satuan`, dan ketiga kolom hasil indexing) menjadi satu kolom `features` menggunakan `VectorAssembler`.


In [4]:
# Membuat StringIndexer untuk nama_menu
indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx")

# Membuat StringIndexer untuk cabang
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")

# Membuat StringIndexer untuk waktu_pesan
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")

# Melatih dan menerapkan StringIndexer
model_menu = indexer_menu.fit(df)
df_indexed = model_menu.transform(df)

model_cabang = indexer_cabang.fit(df_indexed)
df_indexed = model_cabang.transform(df_indexed)

model_waktu = indexer_waktu.fit(df_indexed)
df_indexed = model_waktu.transform(df_indexed)

# Menampilkan hasil indexing
df_indexed.select(
    "nama_menu",
    "nama_menu_idx",
    "cabang",
    "cabang_idx",
    "waktu_pesan",
    "waktu_pesan_idx"
).show(10)

+--------------+-------------+----------+----------+-----------+---------------+
|     nama_menu|nama_menu_idx|    cabang|cabang_idx|waktu_pesan|waktu_pesan_idx|
+--------------+-------------+----------+----------+-----------+---------------+
|Kentang Goreng|          5.0|  Semarang|       2.0|      Malam|            1.0|
|Kentang Goreng|          5.0|Yogyakarta|       1.0|       Sore|            3.0|
|Kentang Goreng|          5.0|  Semarang|       2.0|      Siang|            2.0|
|Kentang Goreng|          5.0|Yogyakarta|       1.0|       Pagi|            0.0|
|    Roti Bakar|          4.0|  Magelang|       0.0|      Malam|            1.0|
|     Americano|          0.0|  Magelang|       0.0|       Pagi|            0.0|
|    Cappuccino|          2.0|  Magelang|       0.0|      Malam|            1.0|
|    Cappuccino|          2.0|  Semarang|       2.0|       Pagi|            0.0|
|     Kopi Susu|          1.0|Yogyakarta|       1.0|       Pagi|            0.0|
|     Americano|          0.

In [5]:
# Membuat VectorAssembler yang menggabungkan 5 fitur
assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)

# Menggabungkan seluruh fitur
df_final = assembler.transform(df_indexed)

# Menampilkan hasil Feature Engineering
df_final.select("features", "total_penjualan").show(10, truncate=False)

+-------------------------+------------------+
|features                 |total_penjualan   |
+-------------------------+------------------+
|[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
|(5,[0,1],[1.0,30000.0])  |32462.966390290698|
|[2.0,25000.0,2.0,0.0,1.0]|51846.84285104993 |
|[5.0,25000.0,2.0,2.0,0.0]|114192.26376783242|
|[1.0,22000.0,1.0,1.0,0.0]|9697.180344260187 |
|(5,[0,1],[2.0,15000.0])  |32316.051747436813|
+-------------------------+------------------+
only showing top 10 rows



**B. Train/Test Split & Pelatihan Model** *(bobot 20%)*

Bagi data menjadi 75% training dan 25% testing (`seed=101`). Latih model `LinearRegression` dengan `labelCol="total_penjualan"`.


In [6]:
# Membagi dataset menjadi training dan testing
train, test = df_final.randomSplit([0.75, 0.25], seed=101)

# Menampilkan jumlah data
print("Jumlah data training:", train.count())
print("Jumlah data testing :", test.count())

Jumlah data training: 1839
Jumlah data testing : 661


In [7]:
# Membuat model Linear Regression
lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")

# Melatih model menggunakan data training
model = lr.fit(train)

print("Model Linear Regression berhasil dilatih!")

# Menampilkan koefisien model dan intercept
print("Koefisien:", model.coefficients)
print("Intercept:", model.intercept)

Model Linear Regression berhasil dilatih!
Koefisien: [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept: -66903.17766485889


**C. Evaluasi Model** *(bobot 20%)*

Hitung dan tampilkan **RMSE** dan **R²** dari model pada data testing. Tulis 1-2 kalimat interpretasi: apakah model ini cukup baik untuk dipakai memprediksi omzet?

In [8]:
# Melakukan prediksi menggunakan data testing
predictions = model.transform(test)

# Menampilkan hasil prediksi
predictions.select(
    "nama_menu",
    "cabang",
    "waktu_pesan",
    "qty",
    "harga_satuan",
    "total_penjualan",
    "prediction"
).show(10, truncate=False)

+---------+--------+-----------+---+------------+------------------+------------------+
|nama_menu|cabang  |waktu_pesan|qty|harga_satuan|total_penjualan   |prediction        |
+---------+--------+-----------+---+------------+------------------+------------------+
|Americano|Magelang|Malam      |1  |25000       |19721.170446051627|31056.146105506836|
|Americano|Magelang|Malam      |1  |25000       |21236.0956379149  |31056.146105506836|
|Americano|Magelang|Malam      |4  |22000       |85834.8564401191  |88127.37331410211 |
|Americano|Magelang|Malam      |4  |22000       |102174.09100317833|88127.37331410211 |
|Americano|Magelang|Malam      |4  |30000       |113044.63431555482|112310.74025938749|
|Americano|Magelang|Malam      |5  |22000       |115810.29600214415|110174.03658512786|
|Americano|Magelang|Malam      |5  |25000       |140386.40273615732|119242.79918960987|
|Americano|Magelang|Pagi       |2  |30000       |73842.64420790318 |67877.77492201277 |
|Americano|Magelang|Pagi       |

In [9]:
# Membuat evaluator RMSE
evaluator_rmse = RegressionEvaluator(
    labelCol="total_penjualan",
    predictionCol="prediction",
    metricName="rmse"
)

# Menghitung RMSE
rmse = evaluator_rmse.evaluate(predictions)

# Membuat evaluator R2
evaluator_r2 = RegressionEvaluator(
    labelCol="total_penjualan",
    predictionCol="prediction",
    metricName="r2"
)

# Menghitung R2
r2 = evaluator_r2.evaluate(predictions)

# Menampilkan hasil evaluasi
print("===== HASIL EVALUASI MODEL UTAMA =====")
print(f"RMSE : Rp {rmse:,.2f}")
print(f"R2   : {r2:.4f}")
print(f"Persentase R2: {r2 * 100:.2f}%")

===== HASIL EVALUASI MODEL UTAMA =====
RMSE : Rp 10,900.80
R2   : 0.9107
Persentase R2: 91.07%


**Interpretasi** : Berdasarkan nilai RMSE dan R2, model ini cukup baik dan layak digunakan untuk memprediksi omzet penjualan yang dibuktikan oleh nilai R2 sebesar 91.07% yang menunjukkan bahwa model mampu menjelaskan sebagian besar variasi penjualan. Sedangkan nilai RMSE sebesar 10,900.80 menunjukkan bahwa masih terdapat selisih antara hasil prediksi dan nilai penjualan sebenarnya.

**D. Eksperimen Perbandingan Fitur** *(bobot 25%)*

Latih **model kedua** yang HANYA menggunakan fitur `qty` dan `harga_satuan` (tanpa informasi menu/cabang/waktu). Bandingkan R² model kedua ini dengan model utama pada bagian C. Tulis kesimpulan: apakah menambahkan informasi menu/cabang/waktu benar-benar meningkatkan akurasi prediksi secara berarti?

In [10]:
# Membuat VectorAssembler untuk model sederhana
assembler_sederhana = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_sederhana"
)

# Menggabungkan dua fitur
df_sederhana = assembler_sederhana.transform(df_indexed)

# Menampilkan hasil
df_sederhana.select("features_sederhana", "total_penjualan").show(10, truncate=False)

+------------------+------------------+
|features_sederhana|total_penjualan   |
+------------------+------------------+
|[1.0,30000.0]     |27750.163531558544|
|[3.0,20000.0]     |64216.16074371864 |
|[5.0,22000.0]     |108722.79259154423|
|[2.0,20000.0]     |54483.205871792496|
|[2.0,30000.0]     |58157.94412015849 |
|[1.0,30000.0]     |32462.966390290698|
|[2.0,25000.0]     |51846.84285104993 |
|[5.0,25000.0]     |114192.26376783242|
|[1.0,22000.0]     |9697.180344260187 |
|[2.0,15000.0]     |32316.051747436813|
+------------------+------------------+
only showing top 10 rows



In [11]:
# Membagi dataset model sederhana
train_sederhana, test_sederhana = df_sederhana.randomSplit([0.75, 0.25], seed=101)

print("Jumlah training sederhana:", train_sederhana.count())
print("Jumlah testing sederhana :", test_sederhana.count())

Jumlah training sederhana: 1839
Jumlah testing sederhana : 661


In [12]:
# Membuat model Linear Regression sederhana
lr_sederhana = LinearRegression(
    featuresCol="features_sederhana",
    labelCol="total_penjualan"
)

# Melatih model kedua
model_sederhana = lr_sederhana.fit(train_sederhana)

print("Model sederhana berhasil dilatih!")

Model sederhana berhasil dilatih!


In [13]:
# Melakukan prediksi pada data testing
prediksi_sederhana = model_sederhana.transform(test_sederhana)

# Membuat evaluator R2
evaluator_r2_sederhana = RegressionEvaluator(
    labelCol="total_penjualan",
    predictionCol="prediction",
    metricName="r2"
)

# Menghitung R2 model sederhana
r2_sederhana = evaluator_r2_sederhana.evaluate(prediksi_sederhana)

print("R2 model sederhana:", round(r2_sederhana, 4))

R2 model sederhana: 0.9108


In [14]:
# Menampilkan perbandingan kedua model
print("===== PERBANDINGAN MODEL =====")

print(f"R2 Model Utama     : {r2:.4f}")
print(f"R2 Model Sederhana : {r2_sederhana:.4f}")

selisih_r2 = r2 - r2_sederhana

print(f"Selisih R2         : {selisih_r2:.4f}")

===== PERBANDINGAN MODEL =====
R2 Model Utama     : 0.9107
R2 Model Sederhana : 0.9108
Selisih R2         : -0.0001


**Kesimpulannya** : Dilihat dari hasil perbandingan, penambahan informasi menu/cabang/waktu pemesanan tidak meningkatkan akurasi prediksi secara berarti. Hal tersebut dapat dilihat dari nilai R2 model utama yaitu sebesar 0.9107, sedangkan R2 model sederhana nilainya sebesar 0.9108. Selisih kedua model hanya -0.0001, sehingga model sederhana memiliki hasil yang sedikit lebih baik. Dengan demikian, fitur qty dan harga_satuan sudah cukup untuk memprediksi total penjualan, sedangkan penambahan informasi menu/cabang/waktu pemesanan belum memberikan peningkatan akurasi yang signifikan.

**E. Prediksi Skenario Baru** *(bobot 15%)*

Buat sebuah DataFrame baru berisi **3 skenario pesanan (data dummy)** (boleh bebas kombinasi menu/cabang/waktu/qty/harga, namun harus melalui tahapan `StringIndexer` dan `VectorAssembler` yang sama seperti data training — perhatikan: gunakan model `StringIndexer` yang **sama** yang sudah di-`fit()` sebelumnya, jangan membuat `StringIndexer` baru). Tampilkan hasil prediksi `total_penjualan` untuk ketiga skenario tersebut.

In [15]:
# Membuat tiga skenario pesanan baru
data_baru = [
    ("Kopi Susu", "Magelang", "Pagi", 2, 20000),
    ("Americano", "Yogyakarta", "Siang", 3, 18000),
    ("Cappuccino", "Semarang", "Malam", 1, 25000)
]

# Membuat DataFrame Spark
kolom = [
    "nama_menu",
    "cabang",
    "waktu_pesan",
    "qty",
    "harga_satuan"
]

df_baru = spark.createDataFrame(data_baru, kolom)

# Menampilkan data pesanan baru
print("DATA PESANAN BARU :")
df_baru.show()

DATA PESANAN BARU :


+----------+----------+-----------+---+------------+
| nama_menu|    cabang|waktu_pesan|qty|harga_satuan|
+----------+----------+-----------+---+------------+
| Kopi Susu|  Magelang|       Pagi|  2|       20000|
| Americano|Yogyakarta|      Siang|  3|       18000|
|Cappuccino|  Semarang|      Malam|  1|       25000|
+----------+----------+-----------+---+------------+



In [16]:
# Menggunakan model StringIndexer yang sudah di-fit sebelumnya
df_baru_indexed = model_menu.transform(df_baru)
df_baru_indexed = model_cabang.transform(df_baru_indexed)
df_baru_indexed = model_waktu.transform(df_baru_indexed)

# Menampilkan hasil indexing
df_baru_indexed.select(
    "nama_menu",
    "nama_menu_idx",
    "cabang",
    "cabang_idx",
    "waktu_pesan",
    "waktu_pesan_idx"
).show()

+----------+-------------+----------+----------+-----------+---------------+
| nama_menu|nama_menu_idx|    cabang|cabang_idx|waktu_pesan|waktu_pesan_idx|
+----------+-------------+----------+----------+-----------+---------------+
| Kopi Susu|          1.0|  Magelang|       0.0|       Pagi|            0.0|
| Americano|          0.0|Yogyakarta|       1.0|      Siang|            2.0|
|Cappuccino|          2.0|  Semarang|       2.0|      Malam|            1.0|
+----------+-------------+----------+----------+-----------+---------------+



In [17]:
# Menggabungkan fitur menggunakan VectorAssembler yang sama
df_baru_final = assembler.transform(df_baru_indexed)

# Menampilkan hasil
df_baru_final.select("features").show(truncate=False)

+-------------------------+
|features                 |
+-------------------------+
|[2.0,20000.0,1.0,0.0,0.0]|
|[3.0,18000.0,0.0,1.0,2.0]|
|[1.0,25000.0,2.0,2.0,1.0]|
+-------------------------+



In [18]:
# Melakukan prediksi menggunakan model utama
hasil_prediksi = model.transform(df_baru_final)

# Menampilkan hasil prediksi
print("HASIL PREDIKSI TOTAL PENJUALAN :")

hasil_prediksi.select(
    "nama_menu",
    "cabang",
    "waktu_pesan",
    "qty",
    "harga_satuan",
    "prediction"
).show(truncate=False)

HASIL PREDIKSI TOTAL PENJUALAN :
+----------+----------+-----------+---+------------+------------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction        |
+----------+----------+-----------+---+------------+------------------+
|Kopi Susu |Magelang  |Pagi       |2  |20000       |37525.35198441948 |
|Americano |Yogyakarta|Siang      |3  |18000       |54607.949890373595|
|Cappuccino|Semarang  |Malam      |1  |25000       |31368.28664276724 |
+----------+----------+-----------+---+------------+------------------+



## Menutup SparkSession

In [19]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
